In [2]:
# ENVIRONMENT SETUP
import sys, subprocess, importlib.util

required = [
    "numpy", "pandas", "matplotlib", "scikit-learn", "xgboost", "shap"
]

missing = [pkg for pkg in required if importlib.util.find_spec(pkg.replace("scikit-learn", "sklearn")) is None]
if missing:
    print("Missing packages:", missing)
else:
    print("All main packages are available.")

All main packages are available.


In [3]:
# IMPORTS
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 100)

# Seasonal Snowfall Prediction in the European Alps
## A Machine Learning Investigation Using Climate Reanalysis and Topographic Data

---

Seasonal snowfall is a mountain-climate problem where **atmospheric variables** and **topography** interact.  
This notebook studies whether **September–November temperature and precipitation**, together with **elevation**, can predict **winter snowfall** in the European Alps.

> Research question: **To what extent can winter snowfall in the European Alps be predicted using autumn climate conditions and geographic characteristics?**

---

## 0. Executive Summary

This project builds and evaluates a supervised machine learning workflow for predicting winter snowfall across Alpine grid cells. The predictors are autumn climate variables from ERA5-Land-style reanalysis data and static topographic elevation. The target variable is seasonal winter snowfall.

The notebook follows a technical-report style:

1. formulate the real-world and mathematical problem;
2. validate and explore the data;
3. engineer physically meaningful features;
4. compare interpretable baseline models with non-linear machine learning models;
5. diagnose model behavior and explain feature effects.

The central hypothesis is that **winter snowfall is controlled by a nonlinear combination of autumn temperature, autumn precipitation, and elevation**. Linear models provide a transparent baseline, while Random Forest and XGBoost are tested because tree ensembles can model interactions and nonlinear thresholds.

---
## Section 1 — Introduction


Seasonal snowfall affects freshwater storage, river discharge, hydropower planning, avalanche risk, ecosystems, and winter tourism. The European Alps are especially sensitive because precipitation phase depends strongly on temperature: small warming can shift precipitation from snow to rain near the freezing point.

### 1.1 Research Question

**To what extent can winter snowfall in the European Alps be predicted using autumn climate conditions and geographic characteristics?**

More specifically:

- Can September–November temperature predict winter snowfall?
- Can autumn precipitation indicate future snow accumulation?
- How important is elevation?
- Which variables contribute most to snowfall variability?
- Can machine learning outperform traditional statistical models?

### 1.2 Objectives

1. Merge climate and elevation data for Alpine grid cells.
2. Validate missingness, plausible ranges, and physical consistency.
3. Build baseline statistical models: Linear Regression, Ridge, and Lasso.
4. Build nonlinear machine learning models: Random Forest and XGBoost.
5. Compare models using MAE, RMSE, and $R^2$.
6. Interpret model decisions using feature importance and SHAP values.

---
## Section 2 — Literature Review

### 2.1 Snowfall Formation

Snowfall forms when atmospheric moisture condenses and freezes into ice crystals, and when the vertical temperature profile allows solid precipitation to reach the surface. Near-surface air temperature is therefore a strong control on whether precipitation falls as rain or snow. In this project, monthly autumn temperature variables are used as predictors because they provide information about seasonal thermal state before winter.

### 2.2 Orographic Precipitation

Mountains force air masses to rise. Rising air cools, reaches saturation, and can produce enhanced precipitation on windward slopes. This process is known as **orographic precipitation**. Studies of mountain precipitation emphasize that terrain lifting can favor condensation and cloud formation, creating strong spatial gradients in precipitation across mountain ranges. Source: Napoli et al. (2019), *Variability of orographic enhancement of precipitation in the Alpine region*, https://pmc.ncbi.nlm.nih.gov/articles/PMC6746858/

### 2.3 Elevation Effects

Elevation affects snowfall through temperature lapse rates and exposure to orographic precipitation. ERA5-Land itself applies an elevation correction to the thermodynamic near-surface state, which shows why elevation is physically important for land-surface climate analysis. Source: Muñoz-Sabater et al. (2021), *ERA5-Land: a state-of-the-art global reanalysis dataset for land applications*, https://essd.copernicus.org/articles/13/4349/2021/

### 2.4 Machine Learning in Climate Science

Machine learning is increasingly used in weather and climate because it can model nonlinear relationships and interactions that are difficult to represent with simple linear equations. Tree-based ensembles are useful for tabular environmental data because they can learn thresholds and variable interactions without requiring explicit polynomial terms.

Useful method references:

- Random Forest Regressor, scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
- Ridge regression, scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html
- Lasso regression, scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html
- XGBoost documentation: https://xgboost.readthedocs.io/en/stable/
- SHAP documentation: https://shap.readthedocs.io/en/latest/

---
## Section 3 — Data Collection

### 3.1 ERA5-Land

The climate dataset represents ERA5-Land-style reanalysis variables over the European Alps:

| Variable | Meaning | Role |
|---|---:|---|
| `t2m` / monthly temperature | 2-meter air temperature | predictor |
| `tp` / monthly precipitation | total precipitation | predictor |
| `sf` / winter snowfall | snowfall | target |

ERA5-Land is produced by ECMWF under the Copernicus Climate Change Service. ECMWF describes ERA5-Land as a high-resolution land reanalysis dataset with hourly surface-variable information and approximately 9 km grid spacing. Source: https://www.ecmwf.int/en/era5-land

The Copernicus Climate Data Store describes ERA5-Land as a dataset providing a consistent view of land-variable evolution over multiple decades. Source: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land

### 3.2 Elevation Dataset

The elevation dataset is a DEM-derived table containing latitude, longitude, and elevation in meters for each grid cell.

### 3.3 Data Processing Pipeline

The data pipeline is:

1. read climate and elevation CSV files;
2. merge by latitude and longitude;
3. validate values;
4. engineer autumn and topographic features;
5. split by year to avoid temporal leakage;
6. train baseline and machine learning models.

In [12]:
# ── LOAD DATA ─────────────────────────────────────────────────────────────────


climate_path = "datasets/alps_snowfall_ml_dataset.csv"
elevation_path = "datasets/alps_elevation.csv"

climate = pd.read_csv(climate_path)
elevation = pd.read_csv(elevation_path)

print("Climate shape:", climate.shape)
print("Elevation shape:", elevation.shape)
display(climate.head())
display(elevation.head())

Climate shape: (147186, 10)
Elevation shape: (5661, 3)


,lat,lon,sep_temp,oct_temp,nov_temp,sep_precip,oct_precip,nov_precip,winter_snowfall,year
0,48.0,5.0,13.720734,9.956451,6.689029,0.093543,0.118970,0.161477,0.022422,2000
1,48.0,5.1,13.533885,9.793838,6.473730,0.096865,0.119708,0.168315,0.023954,2000
2,48.0,5.2,13.359926,9.584475,6.221841,0.098739,0.120086,0.173524,0.024710,2000
3,48.0,5.3,13.320474,9.519834,6.112857,0.096361,0.119496,0.174590,0.023715,2000
4,48.0,5.4,13.381606,9.522227,6.059016,0.092295,0.118310,0.173335,0.021996,2000


,lat,lon,elevation
0,48.0,5.0,312.920105
1,48.0,5.1,353.805573
2,48.0,5.2,402.360992
3,48.0,5.3,380.785126
4,48.0,5.4,462.431274


In [20]:
# ── MERGE CLIMATE + ELEVATION ────────────────────────────────────────────────
for df in [climate, elevation]:
    df["lat_key"] = df["lat"].round(1).astype(str)
    df["lon_key"] = df["lon"].round(1).astype(str)

data = climate.merge(
    elevation[["lat_key", "lon_key", "elevation"]],
    on=["lat_key", "lon_key"],
    how="left"
).drop(columns=["lat_key", "lon_key"])

print("Merged shape:", data.shape)
print("Missing elevation rows:", data["elevation"].isna().sum())
display(data.head())

Merged shape: (147186, 11)
Missing elevation rows: 26858


,lat,lon,sep_temp,oct_temp,nov_temp,sep_precip,oct_precip,nov_precip,winter_snowfall,year,elevation
0,48.0,5.0,13.720734,9.956451,6.689029,0.093543,0.118970,0.161477,0.022422,2000,312.920105
1,48.0,5.1,13.533885,9.793838,6.473730,0.096865,0.119708,0.168315,0.023954,2000,353.805573
2,48.0,5.2,13.359926,9.584475,6.221841,0.098739,0.120086,0.173524,0.024710,2000,402.360992
3,48.0,5.3,13.320474,9.519834,6.112857,0.096361,0.119496,0.174590,0.023715,2000,380.785126
4,48.0,5.4,13.381606,9.522227,6.059016,0.092295,0.118310,0.173335,0.021996,2000,462.431274


In [21]:
print(elevation["elevation"].isna().sum())
print(elevation[elevation["elevation"].isna()].head())

1033
       lat   lon  elevation lat_key lon_key
2634  45.7  13.1        NaN    45.7    13.1
2635  45.7  13.2        NaN    45.7    13.2
2636  45.7  13.3        NaN    45.7    13.3
2637  45.7  13.4        NaN    45.7    13.4
2638  45.7  13.5        NaN    45.7    13.5


---
The elevation dataset contained **1033 grid cells with missing DEM values**. Since these grid cells could not provide valid topographic information, they were removed after merging. This removed 26,858 rows, corresponding to repeated yearly observations for the same missing locations.